# 🧪 W12-D1 概念实验：为什么 MallSenseAI 不是 CV 项目？

> 配套阅读：`第12周-Day1-VisionIntelligence全景.md`（三层架构、ADR-003/004 依据在那边）
> 这个 notebook 用可执行的 mock 管线回答三个问题：
> 1. **检测在整个价值链里占多大比重？** —— 跑通 detector→rules→alerts→工单→通知 全链路
> 2. **客户续费的到底是什么？** —— 告警闭环状态机："堵住 10 分钟内有人处理" vs 事后查录像
> 3. **"封闭系统"封在哪？** —— Capability × Industry 矩阵：没有 capability 注册表意味着什么
>
> 架构映射基于真实仓库 `/root/MallSenseAI`（只读对照，本实验全部用 dataclass 模拟）。
> 实验环境：纯 Python + numpy，不依赖摄像头/模型。

## 实验 1：Detector 层 —— 检测只是管线的第一格

真实架构映射：
- `backend/app/detectors/base.py` → `BaseDetector`（`is_enabled` 开关 + `detect(image, roi_polygons, config)`）
- 检测结果用 normalized `[0,1]` 坐标（与分辨率解耦），**ROI 之外的检测直接丢弃**（ROI=业务关注区）
- 4 个检测器中 `FloorCleanliness` 可被运行时配置关掉 —— `is_enabled` 就是它的位置

本实验：合成 3 个摄像头 × 6 个时刻的物理世界（状态型场景，为什么是截图见 Day2），跑 3 个 mock 检测器。

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
import numpy as np

rng = np.random.default_rng(7)

@dataclass(frozen=True)
class DetectionResult:
    """对应 detectors/base.py 的 DetectionResult（normalized 坐标）"""
    camera_id: str
    label: str
    confidence: float
    polygon: tuple  # (x1, y1, x2, y2)，normalized

    def centroid(self):
        x1, y1, x2, y2 = self.polygon
        return (x1 + x2) / 2, (y1 + y2) / 2

    def inside(self, roi):
        cx, cy = self.centroid()
        rx1, ry1, rx2, ry2 = roi
        return rx1 <= cx <= rx2 and ry1 <= cy <= ry2

class BaseDetector(ABC):
    @property
    @abstractmethod
    def is_enabled(self) -> bool: ...
    @abstractmethod
    def detect(self, camera_id, frame, roi, config) -> list: ...

class DebrisDetector(BaseDetector):
    """对应 debris.py：消防通道障碍物（YOLO）"""
    @property
    def is_enabled(self): return True
    def detect(self, camera_id, frame, roi, config):
        out = []
        for area in frame.get("debris", []):
            conf = float(np.clip(rng.normal(area["conf_mu"], 0.03), 0, 1))
            out.append(DetectionResult(camera_id, "debris", conf, tuple(area["box"])))
        return [d for d in out
                if d.confidence >= config["min_confidence"] and d.inside(roi)]

class FireSmokeDetector(BaseDetector):
    """对应 fire_smoke.py"""
    @property
    def is_enabled(self): return True
    def detect(self, camera_id, frame, roi, config):
        if frame.get("fire"):
            return [DetectionResult(camera_id, "fire_smoke", frame["fire"], (0.2, 0.2, 0.8, 0.8))]
        return []

class FloorCleanlinessDetector(BaseDetector):
    """对应 floor_cleanliness.py：图像对比，当前未启用"""
    @property
    def is_enabled(self): return False   # ← 运行时开关：关掉就不进管线
    def detect(self, camera_id, frame, roi, config):
        return []

# 合成物理世界：3 摄像头 × 6 时刻（每 2h 一张快照）
def synth_world():
    frames = []
    for t in range(6):
        if 2 <= t <= 4:   # 消防通道 t=2~4 被堵 3 个快照（状态型）
            frames.append(("CAM-消防通道L1", t,
                           {"debris": [{"box": (0.30, 0.60, 0.55, 0.85), "conf_mu": 0.90}]}))
        else:
            frames.append(("CAM-消防通道L1", t, {}))
        frames.append(("CAM-中庭L2", t, {"fire": 0.97} if t == 5 else {}))
        if t == 3:        # 货梯厅堆货落在 ROI 之外（轿厢前≠通道区）
            frames.append(("CAM-货梯厅B1", t,
                           {"debris": [{"box": (0.05, 0.05, 0.15, 0.15), "conf_mu": 0.80}]}))
        else:
            frames.append(("CAM-货梯厅B1", t, {}))
    return frames

# ROI 按摄像头配置（normalized）：消防/货梯只关心通道区，中庭全画面
CAM_ROI = {
    "CAM-消防通道L1": (0.25, 0.50, 0.75, 1.00),
    "CAM-货梯厅B1":   (0.30, 0.60, 0.90, 1.00),
    "CAM-中庭L2":     (0.00, 0.00, 1.00, 1.00),
}

DETECTORS = [DebrisDetector(), FireSmokeDetector(), FloorCleanlinessDetector()]
detection_events = []   # 对应 detection_events 审计表（pipeline: capture→detect→persist）

for cam_id, t, frame in synth_world():
    for det in DETECTORS:
        if not det.is_enabled:
            continue    # is_enabled=False 的检测器根本不进管线
        for d in det.detect(cam_id, frame, CAM_ROI[cam_id], {"min_confidence": 0.5}):
            detection_events.append((t, d.camera_id, d.label, round(d.confidence, 2)))

print(f"detection_events 审计记录：{len(detection_events)} 条")
for ev in detection_events:
    print("  t=%dh  %-12s %-10s conf=%.2f" % ev)
print()
print("· 货梯厅B1 t=3 的堆货被丢弃：质心 (0.10,0.10) 落在 ROI 外 —— ROI 是业务关注区，不是画面")
print("· FloorCleanliness is_enabled=False：管线直接跳过（这就是配置开关的位置）")
print("· 到这里为止是“CV 项目”能交付的全部 —— 但 pipeline 还有 rule → alert 两格没走")

## 实验 2：Rules → Alerts —— 规则层把"看见"收敛成"该处理的事"

真实架构映射：
- `backend/app/rules/engine.py`（三模式规则评估）→ 本实验简化为 **置信度阈值 + 冷却窗口**
- `backend/app/rules/cooldown.py` → 同一 camera+rule 在冷却窗口内不重复告警（对应物业 SOP：同一事件只报一次）
- `backend/app/alerts/service.py` → 告警落库 `status=pending`

同一消防通道被堵 3 个快照（6 小时），冷却窗口把 3 条检测收敛成 1 条告警 —— 没有这层，值班员每 2 小时被轰炸一次。

In [ ]:
from dataclasses import dataclass
from collections import Counter

@dataclass
class Alert:
    """对应 models/entities.py 的 Alert（status 默认 pending）"""
    alert_id: int
    camera_id: str
    rule: str
    severity: str
    created_t: int          # 小时
    status: str = "pending"

SEVERITY = {"debris": "high", "fire_smoke": "critical"}
COOLDOWN_H = 3              # 冷却：同 camera+rule 3 小时内不重复（简化自 rules/cooldown.py）

alerts = []
for t, cam_id, label, conf in detection_events:
    in_cooldown = any(a.camera_id == cam_id and a.rule == label
                      and t - a.created_t < COOLDOWN_H for a in alerts)
    if in_cooldown:
        continue
    alerts.append(Alert(len(alerts) + 1, cam_id, label, SEVERITY[label], t))

print("规则层之后：")
for a in alerts:
    print(f"  ALT-{a.alert_id:03d}  t={a.created_t}h  {a.camera_id:<12} rule={a.rule:<10} severity={a.severity:<8} status={a.status}")

print()
print("多场景聚合（DetectionEvent 审计 vs Alert）：")
det_by_cam = Counter(f"{cam}/{lab}" for _, cam, lab, _ in detection_events)
for k, v in sorted(det_by_cam.items()):
    n_alert = sum(1 for a in alerts if f"{a.camera_id}/{a.rule}" == k)
    print(f"  {k:<28} 检测 {v} 条 → 告警 {n_alert} 条")
print()
print(f"合计：{len(detection_events)} 条检测 → {len(alerts)} 条告警")
print("消防通道被堵 3 个快照（同一事件持续存在）只报 1 次 —— 冷却=防轰炸，这就是“规则即 SOP”")

## 实验 3：告警闭环 —— "堵住 10 分钟内有人处理"才是可订阅的产品

真实架构映射：
- `backend/app/models/entities.py` 的 `AlertStatus`：`pending → confirmed → resolved / false_positive`（终态无出边）
- `backend/app/alerts/state_machine.py` 的工单状态机：`open → in_progress → closed / cancelled`，**确认告警后自动建工单**
- 通知渠道（企微/短信/邮件）= 物业值班体系

验证两件事：① 非法迁移被状态机拒绝（pending 不能直接跳 resolved）；② 算一笔闭环时延账 —— 这就是客户续费的理由。

In [ ]:
# 告警状态机（对应 AlertStatus 枚举）
ALERT_TRANSITIONS = {
    "pending":       {"confirmed", "false_positive"},
    "confirmed":     {"resolved"},
    "resolved":      set(),                # 终态
    "false_positive": set(),               # 终态
}
# 工单状态机（对应 alerts/state_machine.py 的 _ALLOWED_TRANSITIONS）
WORKORDER_TRANSITIONS = {
    "open":        {"in_progress", "cancelled"},
    "in_progress": {"closed", "open", "cancelled"},
    "closed":      set(),
    "cancelled":   set(),
}

def transition(obj, to, machine, log):
    allowed = machine[obj["status"]]
    if to not in allowed:
        raise ValueError(f"非法迁移 {obj['status']}→{to}（允许的目标: {sorted(allowed) or ['<终态>']}）")
    obj["status"] = to
    log.append((obj["id"], to))

audit = []   # 运营台账：DetectionEvent → Alert → WorkOrder 全程留痕

# 场景 A：真实告警 —— 确认 → 自动建工单 → 派单 → 销项 → resolved
alert = {"id": "ALT-001", "status": "pending"}
transition(alert, "confirmed", ALERT_TRANSITIONS, audit)
workorder = {"id": "WO-2026-0818-01", "status": "open"}    # 确认即自动创建（工程部派单）
transition(workorder, "in_progress", WORKORDER_TRANSITIONS, audit)
transition(workorder, "closed", WORKORDER_TRANSITIONS, audit)
transition(alert, "resolved", ALERT_TRANSITIONS, audit)

# 场景 B：误报路径 —— pending → false_positive（也是终态留痕）
fp = {"id": "ALT-002", "status": "pending"}
transition(fp, "false_positive", ALERT_TRANSITIONS, audit)

# 反例：想跳过确认直接 resolved —— 被状态机拒绝
try:
    transition({"id": "ALT-003", "status": "pending"}, "resolved", ALERT_TRANSITIONS, audit)
except ValueError as e:
    print("状态机拒绝：", e)

print("\n闭环台账（合规证据链）：")
for obj_id, to in audit:
    print(f"  {obj_id} → {to}")

# 闭环时延账：客户续费的真正理由
confirm_min, dispatch_min, fix_min = 3, 4, 6      # 告警→确认→派单→现场处理
closure = confirm_min + dispatch_min + fix_min
print()
print(f"消防通道被堵 → {closure} 分钟内有人到场处理（每一步在台账里可查）")
print(f"传统 NVR 方案：录像里'看得到'，但没有告警、没有工单、没有销项 —— 检测≠管理")
print(f"结论：CV 交付的是'能识别'，MallSenseAI 交付的是'能闭环' —— 后者才是订阅制的标的")

## 实验 4：Capability × Industry 正交 —— "封闭"封在哪，开放长什么样

md 版第 4 节的关键事实：PRD 与代码双重验证，MallSenseAI **没有任何 capability 目录或注册表**——
上层的 LangChat/OrchestratorAgent 看不见它，只能整包部署。
ADR-003 的解法是正交矩阵：能力名不带行业词（`vision.*`，禁止 `langchat.retail.*`），行业只是应用层标签。

左右对比：同一个能力包 × 不同 industries 标签 = 矩阵中的多个打包结果（制造业产线安全是 P1 适配对象）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import ListedColormap

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

caps = ["vision.detect@v1", "safety.alert.query@v1",
        "safety.alert.subscribe@v1", "vision.rule.configure@v1"]
inds = ["retail\n商场", "manufacturing\n产线", "property\n物业园区"]

closed = np.zeros((len(caps), len(inds)))                 # 现状：capability=∅
target = np.array([[1, 1, 1],
                   [1, 1, 1],
                   [1, 1, 0],
                   [1, 1, 0]], dtype=float)               # 目标态（P1 先覆盖前两行业）

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
panels = [(axes[0], closed, "现状：封闭系统\ncapability=∅，只能整包部署"),
          (axes[1], target, "目标态：LangChat AI Vision\n能力注册 × 行业标签正交复用")]
for ax, mat, title in panels:
    ax.imshow(mat, cmap=ListedColormap(["#f2b8b8", "#bfe3bf"]), vmin=0, vmax=1)
    ax.set_xticks(range(len(inds)))
    ax.set_xticklabels(inds, fontsize=9)
    ax.set_yticks(range(len(caps)))
    ax.set_yticklabels(caps, fontsize=9)
    for i in range(len(caps)):
        for j in range(len(inds)):
            ax.text(j, i, "√" if mat[i, j] else "×", ha="center", va="center", fontsize=12)
    ax.set_title(title, fontsize=10)
plt.tight_layout()
plt.show()

print("· 现状（左）：775+ 测试、40 e2e、真实 4 层商场部署 —— 成熟但不可被编排")
print("· 目标态（右）：同一套 capability，行业只是 industries 标签（retail/manufacturing）")
print("· 更名 MallSenseAI→LangChat AI Vision 只改品牌不动模块名 —— 保护既有契约（ADR-004）")
print("· 思考题的答案方向：B（safety.alert.query/subscribe）为主 + 少量 A；产线巡检还需要 vision.rule.configure")

## 结论：价值链五格，CV 只占第一格

| 环节 | 本实验 | 真实模块 | CV 项目到此为止？ |
|---|---|---|---|
| Detection | 实验 1：ROI 过滤 + is_enabled 开关 | detectors/ | ✅ CV 的终点 |
| Rules | 实验 2：阈值 + 冷却收敛 | rules/ | ❌ 业务从这里开始 |
| Alert 闭环 | 实验 3：状态机 + 台账 + 13 分钟销项 | alerts/ + state_machine | ❌ 订阅价值所在 |
| 工单/通知 | 实验 3：确认即建单 | alerts/service.py | ❌ ERP 人的熟悉领地 |
| Capability 暴露 | 实验 4：正交矩阵（当前=∅） | ——（缺失） | ❌ Week 13 的活 |

**模型会过时，闭环不会。** 客户续费的理由是"消防通道被堵 10 分钟内有人处理"（实验 3），不是"YOLO 精度 92%"（实验 1）。

→ 深入阅读：同目录 `.md` 版本第 3-4 节（三层架构 / ADR-003/004 原文）与第 8 节（Capability 粒度思考题）
→ 明日 Day2：仓库精读 —— 为什么定时截图而不是视频流（有配套实验 notebook）